In [1]:
!pip install -q transformers accelerate sentencepiece bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 19.9 MB/s eta 0:00:00


In [2]:
from huggingface_hub import login
login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
!pip install -U bitsandbytes>=0.46.1

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# Define the quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # or "fp4"
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=bnb_config # Pass the quantization config here
)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [4]:
conversation_history = []

def chat(user_input):
    conversation_history.append(f"User: {user_input}")

    prompt = "\n".join(conversation_history) + "\nAssistant:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

    response = tokenizer.decode(output[0], skip_special_tokens=True)
    assistant_reply = response.split("Assistant:")[-1].strip()

    conversation_history.append(f"Assistant: {assistant_reply}")

    return assistant_reply


In [5]:
print(chat("Hello, who are you?"))
print(chat("Can you explain what risk management is in finance?"))
print(chat("Summarize it in 3 bullet points"))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Of course! Here's a classic: Why don't scientists trust atoms? Because they make up everything! Do you have any specific topic or question I can help with?


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


There are several types of financial risks, but here are some of the most common:
1. Market Risk: This is the risk of loss due to changes in market conditions, including interest rates, exchange rates, and stock prices.
2. Credit Risk: This is the risk that a borrower will default on a loan or other debt obligation.
3. Liquidity Risk: This is the risk that a security or asset cannot be sold at a reasonable price when needed to meet obligations.
4. Operational Risk: This is the risk of loss due
1. Risk management in finance involves identifying, assessing, and mitigating various types of financial risks, such as market, credit, liquidity, and operational risks.
2. Market risk arises from changes in market conditions, credit risk from borrower default, liquidity risk from inability to sell assets, and operational risk from internal processes and external events.
3. Effective risk management strategies include diversification, hedging, and setting risk limits.
